<a href="https://colab.research.google.com/github/RyuJaeJeong/llm-study/blob/master/6_sLLM_%ED%95%99%EC%8A%B5%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers==4.40.1 bitsandbytes==0.43.1 accelerate==0.29.3 datasets==2.19.0 tiktoken==0.6.0 -qqq

In [ ]:
!pip install huggingface_hub==0.22.2
!pip install autotrain-advanced==0.7.77

  Using cached codecarbon-2.3.5-py3-none-any.whl.metadata (6.6 kB)
  Using cached evaluate-0.4.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached ipadic-1.0.0.tar.gz (13.4 MB)
  Preparing metadata (setup.py) ... done
  Using cached jiwer-3.0.3-py3-none-any.whl.metadata (2.6 kB)
  Using cached joblib-1.4.0-py3-none-any.whl.metadata (5.4 kB)
  Using cached loguru-0.7.2-py3-none-any.whl.metadata (23 kB)
  Using cached nltk-3.8.1-py3-none-any.whl.metadata (2.8 kB)
  Using cached optuna-3.6.1-py3-none-any.whl.metadata (17 kB)
  Using cached pillow-10.3.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (9.2 kB)
  Using cached protobuf-4.23.4-cp37-abi3-manylinux2014_x86_64.whl.metadata (540 bytes)
  Using cached sacremoses-0.1.1-py3-none-any.whl.metadata (8.3 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 5.5 MB/s eta 0:00:00
  Using cached werkzeug-3.0.2-py3-none-any.whl.metadata (4.1 kB)
  Using cached xgboost-2.0.3-py3-none-manylinux2014_x86_64.whl.metadata (2.0 kB)
  Using

# SQL 생성 프롬프트

In [ ]:
def make_prompt(ddl, question, query=''):
  prompt = f"""당신은 SQL 봇입니다. DDL의 테이블을 활용한 Question을 해결 할 수 있는 SQL 쿼리를 생성하세요.
  ###DDL:
  {ddl}
  ###Question:
  {question}
  ### SQL:
  {query}"""
  return prompt


# 성능평가 파이프라인

In [ ]:
from pathlib import Path
import json
def make_requests_for_gpt_evaluation(df, filename, dir='requests'):
  if not Path(dir).exists():
    Path(dir).mkdir(parents=True)
  prompts = []
  for idx, row in df.iterrows():
    prompts.append("""
    Based on below DDL and Question, evaluate gen_sql can resolve Question. If gen_sql and gt_sql do equal job, return "yes" else return "no".
    Output JSON Format: {"resolve_yn":""}""" + f"""
      DDL: {row['context']}
      Question: {row['question']}
      gt_sql: {row['answer']}
      gen_sql: {row['gen_sql']}
    """)
  jobs = [{"model": "gpt-4-turbo-preview", "response_format" : {"type": "json_object"}, "messages": [{"role":"system", "content": prompt}]} for prompt in prompts]
  with open(Path(dir, filename), "w") as f:
    for job in jobs:
      json_string = json.dumps(job)
      f.write(json_string+ "\n")

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

"""
비동기 요청 명령
python api_request_parallel_processor.py \
  --requests_filepath {요청 파일 경로} \
  --save_filepath {생성할 결과 파일 경로} \
  --request_url https://api.openai.com/v1/chat/completions \
  --max_requests_per_minute 300 \
  --max_tokens_per_minute 100000 \
  --token_encoding_name cl100k_base \
  --max_attempts 5 \
  --logging_level 20
"""

In [ ]:
import pandas as pd
# 결과 jsonl 파일을 csv로 변환하는 함수
def change_jsonl_to_csv(input_file, output_file, prompt_column="prompt", response_column="response"):
  prompts = []
  responses = []
  with open(input_file, 'r') as json_file:
    for data in json_file:
      prompts.append(json.loads(data)[0]['messages'][0]['content'])
      responses.append(json.loads(data)[1]['choices'][0]['message']['content'])
  df = pd.DataFrame({prompt_column: prompts, response_column: responses})
  df.to_csv(output_file, index=False)
  return df